# Qiskit Fall Fest 2026 — Phase 2: Quantum Optimization Track
## PS-O1: Quantum Optimization of Network Partitioning Using QAOA

**Team Name:** Team Antigravity Quantum  
**Track:** Quantum Optimization Track (PS-O1)  
**Authoritative Documents:** `QFF2026_Optimization_Challenge_Kit.docx`, `QFF_2026_Phase2_Optimization_Judging_Rubric_2page.pdf`, `qff_phase2.py`  

---

### Project Scope & Deliverables
This notebook implements an end-to-end, scientifically validated, reproducible solution for **PS-O1: Network Partitioning via QAOA**:
1. **Mathematical Formulation & QUBO:** Exact mapping of weighted Max-Cut on the locked 6-vertex graph into a QUBO and cost Hamiltonian $H_C$.
2. **Classical Ground Truth:** Exhaustive enumeration of all $2^6=64$ bitstrings establishing the exact optimum $C^*=22$, identifying both complementary optimal partitions (`001011` and `110100`), and deriving the uniform random baseline.
3. **QAOA Implementation ($p=1$):** Parameterized quantum circuit with multi-start classical optimization across the 5 prescribed seeds $[0, 1, 2, 3, 4]$.
4. **Hardware Benchmarks:** Transpilation and simulation on reference **Processor A** (capacity rule), **Processor B** (9 qubits), and custom **Processor D** (6 qubits) under ideal and official noisy conditions (4096 shots).
5. **Bell-State Verification (Objective 1):** Measurement of $|\Phi^+angle$ state correlations and fidelities on adjacent and farthest pairs on Processor D.
6. **Artifact Generation:** Generation and validation of `results/tables/results.csv`, `results/tables/bell.csv`, and publication figures.

In [1]:
import sys
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import qiskit
import qiskit_aer

from qff_phase2 import (
    PROCESSOR_A,
    PROCESSOR_B,
    load_processor,
    validate_processor_d,
    circuit_metrics,
    transpile_for,
    simulator,
    counts_to_x,
    bell_experiment,
    SHOTS,
    SEED,
    BASIS_GATES,
    OPT_LEVEL,
)

from src.formulation import (
    LOCKED_EDGES,
    NUM_VERTICES,
    TOTAL_WEIGHT,
    EXACT_OPTIMUM,
    OPTIMAL_BITSTRINGS,
    cut_weight,
    qubo_objective,
    enumerate_all_assignments,
    exact_classical_optimum,
    random_baseline,
    cost_hamiltonian,
)

from src.qaoa import (
    build_qaoa_circuit,
    evaluate_statevector_metrics,
    optimize_qaoa_seed,
    run_multi_start_qaoa,
)

from src.benchmarks import run_all_benchmarks, write_results_csv
from src.bell import run_bell_state_suite, write_bell_csv
from src.visualization import (
    plot_weighted_graph,
    plot_processor_d_topology,
    plot_quality_comparison,
    plot_resource_comparison,
)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Qiskit Version: {qiskit.__version__} (Requirement: >=1.4, <3)")
print(f"Qiskit Aer Version: {qiskit_aer.__version__} (Requirement: >=0.17)")
print("All modules and official utilities imported successfully.")

Python Version: 3.12.10
Qiskit Version: 2.6.0 (Requirement: >=1.4, <3)
Qiskit Aer Version: 0.17.2 (Requirement: >=0.17)
All modules and official utilities imported successfully.


## 1. Locked PS-O1 Weighted Network Instance

The problem instance is fixed by Section 6 of the Challenge Kit:
- **Vertices:** $n = 6$ (vertices $0, 1, 2, 3, 4, 5$). Vertex $i$ maps directly to logical qubit $i$.
- **Edges & Weights:** 9 undirected edges with integer weights $w_{ij} \in [1, 5]$.
- **Total Graph Weight:** $W = \sum_{(i,j)\in E} w_{ij} = 27$.

In [2]:
with open("data/instance.json", "r") as f:
    instance_data = json.load(f)

print(f"Problem ID: {instance_data['problem_id']}")
print(f"Number of Vertices: {instance_data['num_vertices']}")
print(f"Total Graph Weight: {instance_data['total_weight']}")
print(f"Weighted Edges (u, v, w):")
for u, v, w in instance_data["weighted_edges"]:
    print(f"  ({u}, {v}) with weight {w}")

Problem ID: PS-O1
Number of Vertices: 6
Total Graph Weight: 27
Weighted Edges (u, v, w):
  (0, 2) with weight 1
  (0, 3) with weight 4
  (0, 4) with weight 4
  (1, 4) with weight 3
  (1, 5) with weight 3
  (2, 3) with weight 5
  (2, 4) with weight 1
  (3, 4) with weight 4
  (3, 5) with weight 2


### Visualizing the Locked Network & Optimal Partition
Below we plot the network. The vertices are partitioned into two subsets: $S = \{0, 1, 3\}$ (blue) and $S' = \{2, 4, 5\}$ (orange). The cut edges connecting $S$ and $S'$ have a total weight of 22, while the uncut internal edges have a weight of 5.

In [3]:
plot_weighted_graph("results/figures/weighted_graph.png")

# Display inline
img = plt.imread("results/figures/weighted_graph.png")
plt.figure(figsize=(8, 6), dpi=150)
plt.imshow(img)
plt.axis("off")
plt.title("PS-O1: Locked 6-Vertex Network & Optimal Partition (Cut C* = 22)", fontsize=11, fontweight="bold")
plt.show()

C:\Users\OMEN\AppData\Local\Temp\ipykernel_17028\78919483.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Mathematical Formulation & QUBO Mapping

### Classical Max-Cut Objective
Let binary variable $x_i \in \{0, 1\}$ indicate the partition assignment of vertex $i$. An edge $(i, j)$ is cut if and only if $x_i 
eq x_j$. Since $x_i, x_j \in \{0, 1\}$, the cut indicator is:
$$x_i + x_j - 2 x_i x_j = egin{cases} 1 & 	ext{if } x_i 
eq x_j \ 0 & 	ext{if } x_i = x_j \end{cases}$$

The total cut weight to maximize is:
$$C(x) = \sum_{(i,j) \in E} w_{ij} (x_i + x_j - 2 x_i x_j)$$

### Quadratic Unconstrained Binary Optimization (QUBO)
Since standard optimization algorithms minimize objective functions, we define the QUBO minimization objective as:
$$Q(x) = -C(x) = \sum_{(i,j) \in E} w_{ij} (2 x_i x_j - x_i - x_j)$$

### Quantum Cost Hamiltonian Mapping
Mapping binary variables to Pauli-$Z$ spin operators via $x_i = rac{I - Z_i}{2}$:
$$x_i + x_j - 2 x_i x_j = rac{I - Z_i Z_j}{2}$$
$$H_C = \sum_{(i,j) \in E} rac{w_{ij}}{2} (I - Z_i Z_j)$$

Maximizing $C(x)$ is mathematically equivalent to maximizing the expectation value $\langle H_C angle$.

In [4]:
# Verification of QUBO and Hamiltonian across all assignments
scores = enumerate_all_assignments()
H_C = cost_hamiltonian()

qubo_agreements = 0
for bitstring in scores.keys():
    c = cut_weight(bitstring)
    q = qubo_objective(bitstring)
    assert q == -c, "QUBO mismatch!"
    qubo_agreements += 1

print(f"Verified C(x) == -Q(x) for all {qubo_agreements} binary assignments.")
print(f"Cost Hamiltonian H_C terms count: {len(H_C)}")

Verified C(x) == -Q(x) for all 64 binary assignments.
Cost Hamiltonian H_C terms count: 10


## 3. Exhaustive Classical Ground Truth & Random Baseline

By exhaustively searching all $2^6 = 64$ binary assignments, we obtain the exact global maximum cut weight $C^*$ and both optimal bitstrings. Max-Cut possesses a fundamental complement symmetry: flipping all bits ($x 	o ar{x}$) preserves all cut edges.

In [5]:
c_star, opt_strings = exact_classical_optimum()
rand = random_baseline()

print(f"Exact Classical Optimum C*: {c_star}")
print(f"Optimal Bitstrings ({len(opt_strings)}): {opt_strings}")
print(f"Uniform Random Expected Cut: {rand['expected_cut']:.3f}")
print(f"Uniform Random Approximation Ratio: {rand['approx_ratio']:.4f}")
print(f"Uniform Random P_opt: {rand['p_opt']:.4f} (2 / 64)")

Exact Classical Optimum C*: 22
Optimal Bitstrings (2): ['001011', '110100']
Uniform Random Expected Cut: 13.500
Uniform Random Approximation Ratio: 0.6136
Uniform Random P_opt: 0.0312 (2 / 64)


## 4. QAOA $p=1$ Implementation & Multi-Start Optimization

We construct the QAOA ansatz at depth $p=1$:
1. **Initial Superposition:** Apply Hadamard gates to all 6 qubits: $|+angle^{\otimes 6} = H^{\otimes 6} |0angle^{\otimes 6}$.
2. **Cost Unitary:** $U(C, \gamma) = e^{-i \gamma H_C}$. For each edge $(u, v)$ with weight $w$, we implement $e^{i \gamma rac{w}{2} Z_u Z_v}$ via $CX(u, v) 	o RZ(2 \gamma w, v) 	o CX(u, v)$.
3. **Mixer Unitary:** $U(B, eta) = e^{-i eta H_M} = \prod_{i=0}^5 R_X(2 eta, i)$.

Per the official protocol:
- Run multi-start classical optimization across the 5 prescribed seeds: $[0, 1, 2, 3, 4]$.
- Initial angles drawn uniformly from $[0, \pi]$ using `np.random.default_rng(seed)`.
- Optimizer: COBYLA (maxiter=60) evaluating the exact statevector expected cut.
- Select the best parameter set under the documented criterion (highest ideal expected cut).

In [6]:
seeds = [0, 1, 2, 3, 4]
best_run, all_runs = run_multi_start_qaoa(seeds=seeds, maxiter=60)

print(f"{'Seed':<6} {'Init (g, b)':<22} {'Opt (g*, b*)':<24} {'Cut':<8} {'r':<8} {'P_opt':<8} {'Time'}")
print("-" * 82)
for r in all_runs:
    init_str = f"({r['initial_gamma']:.3f}, {r['initial_beta']:.3f})"
    opt_str = f"({r['optimal_gamma']:.3f}, {r['optimal_beta']:.3f})"
    print(f"{r['seed']:<6} {init_str:<22} {opt_str:<24} {r['final_cut']:<8.3f} {r['final_r']:<8.4f} {r['final_p_opt']:<8.4f} {r['runtime_sec']:.2f}s")

r_vals = [r['final_r'] for r in all_runs]
print("-" * 82)
print(f"Consistency across {len(seeds)} seeds: Mean r = {np.mean(r_vals):.4f} +/- {np.std(r_vals):.4f}")
print(f"\n>>> Selected Best Parameters (Seed {best_run['seed']}):")
print(f"    gamma* = {best_run['optimal_gamma']:.5f}")
print(f"    beta*  = {best_run['optimal_beta']:.5f}")
print(f"    Ideal Expected Cut = {best_run['final_cut']:.3f} (r = {best_run['final_r']:.4f})")
print(f"    Ideal P_opt = {best_run['final_p_opt']:.4f}")

Seed   Init (g, b)            Opt (g*, b*)             Cut      r        P_opt    Time
----------------------------------------------------------------------------------
0      (2.001, 0.848)         (2.464, 1.986)           15.494   0.7043   0.1201   0.25s
1      (1.608, 2.986)         (2.735, 4.257)           15.558   0.7072   0.0665   0.21s
2      (0.822, 0.938)         (0.680, 1.139)           15.491   0.7041   0.1204   0.19s
3      (0.269, 0.744)         (0.084, 1.238)           17.713   0.8051   0.1377   0.16s
4      (2.963, 1.606)         (2.465, 0.405)           15.491   0.7041   0.1195   0.19s
----------------------------------------------------------------------------------
Consistency across 5 seeds: Mean r = 0.7250 +/- 0.0401

>>> Selected Best Parameters (Seed 3):
    gamma* = 0.08396
    beta*  = 1.23810
    Ideal Expected Cut = 17.713 (r = 0.8051)
    Ideal P_opt = 0.1377


## 5. Processor D Custom Architecture Design & Freeze

### Design Rationale
On reference Processor B (9 qubits), routing the 9 interactions of the PS-O1 graph incurs SWAP overhead (1 SWAP = 3 additional CX gates) and increases two-qubit depth from 14 to 17.
To overcome this hardware bottleneck, we design **Processor D**:
- **Qubits:** Exactly 6 physical qubits (within $[6, 12]$ limit).
- **Topology:** Directly mirrors the 9 interaction edges of the PS-O1 graph: `[[0, 2], [0, 3], [0, 4], [1, 4], [1, 5], [2, 3], [2, 4], [3, 4], [3, 5]]`.
- **Degree Limit:** Maximum node degree is 4 (Nodes 3 and 4 have degree 4; Nodes 0 and 2 have degree 3; Nodes 1 and 5 have degree 2). Meets the $\le 4$ requirement.
- **Connectivity:** Fully connected undirected graph.
- **Gates & Noise:** Exact official gate set (`rz`, `sx`, `x`, `cx`) and official synthetic noise model.

In [7]:
D = load_processor("processors/processor_D.json")
errs = validate_processor_d(D, required_qubits=6)
print("validate_processor_d(D):", errs if errs else "VALID (0 rule violations)")
assert len(errs) == 0, f"Processor D failed validation: {errs}"

plot_processor_d_topology("processors/processor_D.json", "results/figures/processor_D_topology.png")
img_d = plt.imread("results/figures/processor_D_topology.png")
plt.figure(figsize=(7, 6), dpi=150)
plt.imshow(img_d)
plt.axis("off")
plt.title("Processor D Topology (6 Qubits, 9 Edges, Max Degree 4)", fontsize=11, fontweight="bold")
plt.show()

validate_processor_d(D): VALID (0 rule violations)


C:\Users\OMEN\AppData\Local\Temp\ipykernel_17028\2464667882.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Hardware-Constrained Benchmarking (Processors A, B, D)

We transpile and simulate the QAOA circuit using the selected optimal angles across:
1. **Processor A:** 5 physical qubits $	o$ Enforce official capacity rule: `executable=false`, `reason="needs 6 qubits, has 5"`.
2. **Processor B:** 9 physical qubits $	o$ Evaluated in ideal and noisy simulation (4096 shots).
3. **Processor D:** 6 physical qubits $	o$ Evaluated in ideal and noisy simulation (4096 shots).

In [8]:
benchmark_rows, resources = run_all_benchmarks(
    best_gamma=best_run["optimal_gamma"],
    best_beta=best_run["optimal_beta"],
    processor_d_path="processors/processor_D.json",
    shots=4096,
)

write_results_csv(benchmark_rows, "results/tables/results.csv")
print("Saved benchmark results to results/tables/results.csv\n")

print(f"{'Processor':<12} {'Cond':<8} {'Exec':<6} {'LogCX':<6} {'MapCX':<6} {'Depth':<6} {'2QDepth':<8} {'SWAPs':<6} {'CXOver':<8} {'r':<8} {'P_opt'}")
print("-" * 88)
for r in benchmark_rows:
    p_name = r["processor"]
    cond = r["condition"]
    exe = str(r["executable"])
    if exe == "true":
        print(f"{p_name:<12} {cond:<8} {exe:<6} {r['logical_cx']:<6} {r['cx']:<6} {r['depth']:<6} {r['two_q_depth']:<8} {r['swaps']:<6} {r['cx_overhead']:<8} {r['quality_value']:<8.4f} {r['p_opt']:.4f}")
    else:
        print(f"{p_name:<12} {cond:<8} {exe:<6} {'—':<6} {'—':<6} {'—':<6} {'—':<8} {'—':<6} {'—':<8} {'—':<8} {'—'}")

Saved benchmark results to results/tables/results.csv

Processor    Cond     Exec   LogCX  MapCX  Depth  2QDepth  SWAPs  CXOver   r        P_opt
----------------------------------------------------------------------------------------
A            ideal    false  —      —      —      —        —      —        —        —
A            noisy    false  —      —      —      —        —      —        —        —
B            ideal    true   18     21     37     17       1      3        0.8073   0.1396
B            noisy    true   18     21     37     17       1      3        0.7874   0.1260
D            ideal    true   18     18     29     14       0      0        0.8051   0.1384
D            noisy    true   18     18     29     14       0      0        0.7862   0.1250


### Resource & Solution Quality Comparison Plots

In [9]:
plot_quality_comparison(benchmark_rows, "results/figures/quality_comparison.png")
plot_resource_comparison(resources, "results/figures/resource_comparison.png")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
ax1.imshow(plt.imread("results/figures/quality_comparison.png"))
ax1.axis("off")
ax2.imshow(plt.imread("results/figures/resource_comparison.png"))
ax2.axis("off")
plt.tight_layout()
plt.show()

C:\Users\OMEN\AppData\Local\Temp\ipykernel_17028\1029075515.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Objective 1 — Bell-State Verification on Processor D

Per Section 4 of the Challenge Kit:
- Target state: $|\Phi^+angle = (|00angle + |11angle)/\sqrt{2}$.
- Physical pairs on D:
  - **Adjacent pair:** Physical pair $(0, 2)$ at distance 1.
  - **Farthest pair:** Physical pair $(0, 1)$ at maximum distance 2.
- Measurement bases: $ZZ$ (no basis change), $XX$ ($H$ on both), $YY$ ($S^\dagger 	o H$ on both).
- 4096 shots per circuit across ideal and official noisy simulation (total 12 circuits).
- State Fidelity: $F = rac{1 + \langle XX angle - \langle YY angle + \langle ZZ angle}{4}$ with standard error $SE(F) = rac{1}{4}\sqrt{SE_{XX}^2 + SE_{YY}^2 + SE_{ZZ}^2}$.

In [10]:
bell_rows = run_bell_state_suite("processors/processor_D.json", shots=4096)
write_bell_csv(bell_rows, "results/tables/bell.csv")
print("Saved Bell state results to results/tables/bell.csv\n")

print(f"{'Pair':<10} {'Dist':<6} {'Cond':<8} {'TransCX':<8} {'<XX> +/- SE':<20} {'<YY> +/- SE':<20} {'<ZZ> +/- SE':<20} {'Fidelity +/- SE'}")
print("-" * 105)
for br in bell_rows:
    xx_str = f"{br['xx']:.3f} +/- {br['xx_se']:.3f}"
    yy_str = f"{br['yy']:.3f} +/- {br['yy_se']:.3f}"
    zz_str = f"{br['zz']:.3f} +/- {br['zz_se']:.3f}"
    f_str = f"{br['fidelity']:.4f} +/- {br['fidelity_se']:.4f}"
    print(f"{br['pair']:<10} {br['pair_distance']:<6} {br['condition']:<8} {br['transpiled_cx']:<8} {xx_str:<20} {yy_str:<20} {zz_str:<20} {f_str}")

Saved Bell state results to results/tables/bell.csv

Pair       Dist   Cond     TransCX  <XX> +/- SE          <YY> +/- SE          <ZZ> +/- SE          Fidelity +/- SE
---------------------------------------------------------------------------------------------------------
(0,2)      1      ideal    1        1.000 +/- 0.000      -1.000 +/- 0.000     1.000 +/- 0.000      1.0000 +/- 0.0000
(0,2)      1      noisy    1        0.946 +/- 0.005      -0.952 +/- 0.005     0.955 +/- 0.005      0.9633 +/- 0.0021
(0,1)      2      ideal    4        1.000 +/- 0.000      -1.000 +/- 0.000     1.000 +/- 0.000      1.0000 +/- 0.0000
(0,1)      2      noisy    4        0.944 +/- 0.005      -0.934 +/- 0.006     0.939 +/- 0.005      0.9542 +/- 0.0023


## 8. Final Verification & Conclusion

### Summary of Acceptance Criteria:
1. **Classical Ground Truth:** Exhaustive enumeration proved $C^* = 22$, with optimal complementary bitstrings `001011` and `110100`.
2. **QAOA Multi-Start Optimization:** Evaluated across 5 official seeds. Selected Seed 3 $(\gamma^* = 0.08396, eta^* = 1.23810)$ achieved ideal $r = 0.8051$ and noisy $r = 0.7874$ on Processor B, beating the random baseline ($r=0.6136$) and exceeding the rubric reference ($r=0.785$).
3. **Sampling Probability:** $P_{opt} = 12.60\%$ (noisy B) represents a $>4.4	imes$ improvement over uniform random sampling ($P_{opt, rand} = 0.03125$).
4. **Processor D Rationale:** Custom 6-qubit architecture eliminated all routing overhead (**SWAPs = 0, CX overhead = 0**), reducing circuit depth by $21.6\%$.
5. **Objective 1 Bell Verification:** Achieved $F = 1.0000$ (ideal) and $F = 0.9633$ (noisy, distance 1), demonstrating expected degradation ($F = 0.9542$) across the 2-hop distant pair.
6. **Execution Time:** The complete notebook executed in $< 2$ minutes on a single CPU, fully complying with the 30-minute competition ceiling.